# Import Fabric Ontology from Lakehouse

Reads an exported ontology definition JSON from a lakehouse Files folder and creates a new ontology in the target workspace.

**What it does:**
1. Reads `*_definition.json` from the configured lakehouse path
2. Validates structure and shows entity/binding summary
3. Rewrites data-source bindings to point at the target lakehouse
4. Creates the ontology (with retry for name conflicts after deletion)
5. Verifies the result

**Prerequisites:**
- Attach the lakehouse which contains the exported `*_definition.json` to the notebook
- Run the **Export** notebook first, then copy the exported `*_definition.json` to the import   
  lakehouse Files folder
- Create a folder 'FabricIQ-export_import_package' under Files section of your lakehouse. 
- Copy 'fabric_ontology_import-1.0.0-py3-none-any.whl' in that folder
- Install the `fabric-ontology-import` wheel in your Fabric environment (or run the `%pip install` 
  cell below)

## 1. Install Package (skip if already in your Fabric Environment)

Uncomment and update the path to your `.whl` file.

In [1]:
# Uncomment the line below and update the path to your .whl file location
%pip install /lakehouse/default/Files/FabricIQ-export_import_package/fabric_ontology_import-1.0.0-py3-none-any.whl

StatementMeta(, 4d1beab6-9e47-4548-a901-e4e6f0acc20d, 8, Finished, Available, Finished, False)

Processing /lakehouse/default/Files/FabricIQ-export_import_package/fabric_ontology_import-1.0.0-py3-none-any.whl

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



## 2. Configuration

**Lakehouse path format:** `abfss://<WorkspaceName>@onelake.dfs.fabric.microsoft.com/<LakehouseName>.Lakehouse/Files`

Copy the exported `*_definition.json` file to the lakehouse Files folder, then set `DEFINITION_PATH` to point at it.

> **Note:** Source item display names are embedded in the definition JSON at export time, so no access to the source workspace is needed.

In [6]:
# ── Where to find the exported definition ────────────────────
WORKSPACE_NAME      = "WS_AutoClaimsPOC"
LAKEHOUSE_NAME      = "lh_imported_ontology"
EXPORT_FOLDER       = "ontology_exports"
SOURCE_ONTOLOGY     = "ont_UBI01"

DEFINITION_PATH = (
    f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com"
    f"/{LAKEHOUSE_NAME}.Lakehouse/Files/{EXPORT_FOLDER}"
    f"/{SOURCE_ONTOLOGY}_definition.json"
)

# ── Target workspace & new ontology name ─────────────────────
TARGET_WORKSPACE_ID = "db7dcf85-001e-4277-a85e-3c92029900bc"
NEW_ONTOLOGY_NAME   = "ont_UBI02"
DESCRIPTION         = "Created from exported definition of ont_UBI01"

# ── Target data sources (for binding rewrite) ────────────────
# The ontology's data bindings will be rewritten to point at these items.
# Leave a list empty [] to skip that source type.

#***IMPORTANT***
#Ensure the underlying delta tables within the lakehouses or warehouses and kql tables within the eventhouses/kql databases are 
#created with the same names as the source ontology for bindings to work, else entities will be unbounded
#***IMPORTANT***

TARGET_LAKEHOUSE_NAMES      = ["lh_imported_ontology"]
TARGET_WAREHOUSE_NAMES      = []
TARGET_EVENTHOUSE_NAMES     = ["ev_driver_telemetry_imported_ontology"]
TARGET_SEMANTIC_MODEL_NAMES = []

# ── Behavior ─────────────────────────────────────────────────
OVERWRITE_IF_EXISTS     = True
SKIP_BINDING_VALIDATION = False

print(f"Definition:  {DEFINITION_PATH}")
print(f"Target:      {NEW_ONTOLOGY_NAME} in workspace {TARGET_WORKSPACE_ID}")
print(f"Lakehouses:  {TARGET_LAKEHOUSE_NAMES}")

StatementMeta(, 4d1beab6-9e47-4548-a901-e4e6f0acc20d, 14, Finished, Available, Finished, False)

Definition:  abfss://WS_AutoClaimsPOC@onelake.dfs.fabric.microsoft.com/lh_imported_ontology.Lakehouse/Files/ontology_exports/ont_UBI01_definition.json
Target:      ont_UBI02 in workspace db7dcf85-001e-4277-a85e-3c92029900bc
Lakehouses:  ['lh_imported_ontology']


## 3. Import

In [7]:
from fabric_ontology_import import import_ontology

token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")

result = import_ontology(
    token=token,
    definition_path=DEFINITION_PATH,
    target_workspace_id=TARGET_WORKSPACE_ID,
    new_ontology_name=NEW_ONTOLOGY_NAME,
    description=DESCRIPTION,
    target_lakehouse_names=TARGET_LAKEHOUSE_NAMES,
    target_warehouse_names=TARGET_WAREHOUSE_NAMES,
    target_eventhouse_names=TARGET_EVENTHOUSE_NAMES,
    target_semantic_model_names=TARGET_SEMANTIC_MODEL_NAMES,
    overwrite=OVERWRITE_IF_EXISTS,
    skip_binding_validation=SKIP_BINDING_VALIDATION,
)

print(f"\nOntology ID: {result['ontology_id']}")
print(f"Bindings rewritten: {result['rewrite_count']}")
print(f"Bindings dropped (unbound): {len(result['dropped_bindings'])}")
print(f"Verification: {result['verification']['status']}")

StatementMeta(, 4d1beab6-9e47-4548-a901-e4e6f0acc20d, 15, Finished, Available, Finished, False)

Validating target workspace db7dcf85-001e-4277-a85e-3c92029900bc ...
  Workspace: WS_AutoClaimsPOC

Reading definition from: abfss://WS_AutoClaimsPOC@onelake.dfs.fabric.microsoft.com/lh_imported_ontology.Lakehouse/Files/ontology_exports/ont_UBI01_definition.json
  Loaded 30 part(s)
  Source-item map: 2 item(s) embedded

 DEFINITION SUMMARY
 Total parts:         30
 Entity Types:        8
 Relationship Types:  6
 Data Bindings:       8
 Contextualizations:  6
   Policyholder  (8 props, 0 timeseries)
   Policy  (7 props, 0 timeseries)
   Vehicle  (4 props, 0 timeseries)
   Claim  (8 props, 0 timeseries)
   Adjuster  (4 props, 0 timeseries)
   Accident  (7 props, 0 timeseries)
   Trip_Telemetry  (3 props, 19 timeseries)
   ?  (0 props, 0 timeseries)

Found 14 binding(s), 7 unique table(s):

 Owner                               Type                 Source             Table
 ─────────────────────────────────── ──────────────────── ────────────────── ─────────────────────────
 Entity/1141540